# 03 — Mamba SSM v2: V1+V4 Feature Pool + 4-Stage Selection

**Task ID:** `MODEL-MAMBA-ZERO-FEE-006`  
**Version:** 0.2 — Full feature pool from LGBM + 4-stage selection  
**Date:** 2026-05-31

## Pipeline
```
V1 (195 features)  +  V4 (25 features)
         │
         ▼
  4-Stage Feature Selection  (fit on train only)
  Stage 1: variance drop + Spearman collinearity (ρ > 0.85)
  Stage 2: Mutual Information ranking → top 60
  Stage 3: multi-window MI stability (3 sub-windows, majority vote)
  Stage 4: LGBM permutation pruning → keep ~15
         │
         ▼
  Mamba SSM (d_model=64, 2 layers, d_state=16)
  Trained on 4 subsets × 2 labels = 8 runs
  + Permutation Importance + Integrated Gradients
```

## Sequence mechanics
- **Interval:** 1-hour bars  
- **Sequence length:** 24 bars = 24 hours  
- **Train stride:** 1 bar  
- **Normalizer:** QuantileTransformer fitted on train only

In [1]:
import json
import math
import time
import warnings
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import QuantileTransformer
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='darkgrid', palette='muted', font_scale=1.0)

if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    print('Device: Apple MPS')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f'Device: CUDA ({torch.cuda.get_device_name()})')
else:
    DEVICE = torch.device('cpu')
    print('Device: CPU')

Device: Apple MPS


In [2]:
import sys

def _find_repo_root() -> Path:
    p = Path.cwd()
    while p != p.parent:
        if (p / 'pyproject.toml').exists():
            return p
        p = p.parent
    raise RuntimeError('pyproject.toml not found')

REPO     = _find_repo_root()
RAW_DIR  = REPO / 'data' / 'raw'
FEAT_DIR = REPO / 'data' / 'features'
ARTS_DIR = REPO / 'artifacts' / '03_mamba_omni_0fee_v2'
ARTS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO / 'src'))

print(f'Repo: {REPO}')

# ── Sequence constants ────────────────────────────────────────────────────────
SYMBOL      = 'BTCUSDT'
INTERVAL    = '1h'
SEQ_LEN     = 24      # 24h context at 1h bars
STRIDE      = 1       # 1-bar step
BATCH       = 256
EPOCHS      = 50
LR          = 3e-4
FOCAL_GAMMA = 2.0

# Feature selection params (mirrors LGBM 4-stage pipeline)
STAGE2_TOP_K        = 60
STAGE3_N_SUBWINDOWS = 3
STAGE3_MIN_WINS     = 2
STAGE4_TOP_K        = 20   # slightly more than LGBM (15) — Mamba benefits from more context

# Splits (same as LGBM v8)
OOS_START = pd.Timestamp('2024-01-01')
TRAIN_END = pd.Timestamp('2022-12-31')
VAL_END   = pd.Timestamp('2023-12-31')

SELECTION_CACHE = ARTS_DIR / 'selected_features.json'

print(f'SEQ_LEN={SEQ_LEN}  STRIDE={STRIDE}  BATCH={BATCH}  EPOCHS={EPOCHS}')
print(f'OOS from {OOS_START.date()}  |  Train≤{TRAIN_END.date()}  Val≤{VAL_END.date()}')

Repo: /Users/wojciech.neuman/Documents/hybrid-multi-agent-trading-system
SEQ_LEN=24  STRIDE=1  BATCH=256  EPOCHS=50
OOS from 2024-01-01  |  Train≤2022-12-31  Val≤2023-12-31


---
## Phase A — Load V1 + V4 Features

Mirrors LGBM v8: V1 features from `BTCUSDT_1h_features.parquet` (195 structural/TA features) merged with V4 microstructure features (`BTCUSDT_1h_v4_features.parquet`). No computation needed — both are pre-built.

In [3]:
# ── V4: microstructure + regime features ──────────────────────────────────────
v4_df = pd.read_parquet(FEAT_DIR / 'BTCUSDT_1h_v4_features.parquet')
v4_df.index = v4_df.index.tz_localize(None) if v4_df.index.tz else v4_df.index
V4_FEATURE_COLS = list(v4_df.columns)
print(f'V4: {len(V4_FEATURE_COLS)} features  ({v4_df.index.min().date()} → {v4_df.index.max().date()})')

# ── V1: structural + TA features ─────────────────────────────────────────────
v1_df = pd.read_parquet(FEAT_DIR / 'BTCUSDT_1h_features.parquet')
v1_df.index = v1_df.index.tz_localize(None) if v1_df.index.tz else v1_df.index

with open(FEAT_DIR / 'feature_registry.json') as f:
    registry = json.load(f)
BACKTEST_COLS = registry.get('backtest_only_cols', [])

_EXCLUDE = set(BACKTEST_COLS) | {
    'open', 'high', 'low', 'close', 'volume',
    'label', 'tbm_label', 'fh_label', 'forward_ret',
}
V1_FEATURE_COLS = [
    c for c in v1_df.columns
    if c not in _EXCLUDE and pd.api.types.is_numeric_dtype(v1_df[c])
]
print(f'V1: {len(V1_FEATURE_COLS)} features  ({v1_df.index.min().date()} → {v1_df.index.max().date()})')

# ── Raw OHLCV (for ATR / TBM labels) ─────────────────────────────────────────
raw_df = pd.read_parquet(RAW_DIR / 'BTCUSDT_1h.parquet',
                          columns=['open', 'high', 'low', 'close', 'volume'])
raw_df.index = raw_df.index.tz_localize(None) if raw_df.index.tz else raw_df.index

# ── Merge V1 + V4 ─────────────────────────────────────────────────────────────
merged = v1_df.copy()
for col in V4_FEATURE_COLS:
    merged[col] = v4_df[col].reindex(merged.index)
merged['high']  = raw_df['high'].reindex(merged.index)
merged['low']   = raw_df['low'].reindex(merged.index)
merged['close'] = raw_df['close'].reindex(merged.index)
merged.index    = merged.index.tz_localize(None) if merged.index.tz else merged.index

ALL_FEATURE_COLS = V1_FEATURE_COLS + V4_FEATURE_COLS

print(f'\nMerged: {len(merged):,} rows  |  {len(ALL_FEATURE_COLS)} features (V1+V4)')
print(f'  Range: {merged.index.min().date()} → {merged.index.max().date()}')

V4: 25 features  (2017-11-15 → 2026-05-16)
V1: 195 features  (2017-11-15 → 2026-05-16)

Merged: 74,366 rows  |  220 features (V1+V4)
  Range: 2017-11-15 → 2026-05-16


---
## Phase B — 4-Stage Feature Selection

Identical pipeline to LGBM v8, fitted **on train split only** (no OOS contamination):

| Stage | Method | Output |
|-------|--------|--------|
| 1 | Variance drop + Spearman collinearity (ρ > 0.85) | survivors |
| 2 | Mutual Information ranking | top 60 |
| 3 | Multi-window MI stability (3 sub-windows, ≥2 wins) | stable subset |
| 4 | LGBM permutation pruning | top 20 |

Result cached to `artifacts/.../selected_features.json`.

In [4]:
# ── Train split for selection (pre-OOS, same cutoff as LGBM v8) ───────────────
train_sel_mask = merged.index < OOS_START
df_sel = merged[train_sel_mask][ALL_FEATURE_COLS].copy()
# Use 1h forward return as the proxy label for feature selection
y_sel  = (raw_df['close'].reindex(merged.index).pct_change(1).shift(-1)[train_sel_mask] > 0).astype(int).values
X_sel  = df_sel.fillna(0).values.astype(np.float32)
print(f'Selection set: {X_sel.shape[0]:,} bars × {X_sel.shape[1]} features')


# ─────────────────────────────────────────────────────────────────────────────
def _stage1(X, cols, var_thresh=1e-6, rho_thresh=0.85):
    from scipy.stats import spearmanr
    # variance drop
    keep = [c for c, v in zip(cols, X.var(axis=0)) if v > var_thresh]
    idx  = [cols.index(c) for c in keep]
    X1   = X[:, idx]
    # Spearman collinearity
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        corr, _ = spearmanr(X1)
    if X1.shape[1] == 1:
        return keep, X1
    corr = np.abs(np.array(corr))
    np.fill_diagonal(corr, 0.0)
    removed = set()
    for i in range(len(keep)):
        if i in removed: continue
        for j in range(i + 1, len(keep)):
            if j not in removed and corr[i, j] > rho_thresh:
                removed.add(j)
    survive = [c for i, c in enumerate(keep) if i not in removed]
    idx2    = [keep.index(c) for c in survive]
    return survive, X1[:, idx2]


def _stage2(X, y, cols, top_k=STAGE2_TOP_K):
    mi     = mutual_info_classif(X, y, discrete_features=False, random_state=42)
    ranked = sorted(zip(cols, mi), key=lambda t: t[1], reverse=True)
    top    = [c for c, _ in ranked[:top_k]]
    idx    = [cols.index(c) for c in top]
    return top, X[:, idx]


def _stage3(X, y, cols, n_win=STAGE3_N_SUBWINDOWS, min_wins=STAGE3_MIN_WINS,
            top_k=STAGE2_TOP_K):
    n      = len(X)
    sub_sz = n // n_win
    counts = {c: 0 for c in cols}
    for w in range(n_win):
        sl = slice(w * sub_sz, (w + 1) * sub_sz if w < n_win - 1 else n)
        Xw, yw = X[sl], y[sl]
        if len(np.unique(yw)) < 2: continue
        mi  = mutual_info_classif(Xw, yw, discrete_features=False, random_state=42)
        top = sorted(zip(cols, mi), key=lambda t: t[1], reverse=True)[:top_k]
        for c, _ in top: counts[c] += 1
    survive = [c for c in cols if counts[c] >= min_wins]
    if len(survive) < STAGE4_TOP_K:
        survive = cols   # fall back
    idx = [cols.index(c) for c in survive]
    return survive, X[:, idx]


def _logloss(probs, y):
    probs = np.clip(probs, 1e-7, 1.0 - 1e-7)
    return float(-np.mean(y * np.log(probs[:, 1]) + (1 - y) * np.log(probs[:, 0])))


def _stage4(X, y, cols, top_k=STAGE4_TOP_K, n_repeats=3):
    n_val = max(int(0.10 * len(X)), 50)
    X_tr, y_tr = X[:-n_val], y[:-n_val]
    X_vl, y_vl = X[-n_val:], y[-n_val:]
    params = dict(objective='binary', metric='binary_logloss', n_estimators=200,
                  num_leaves=31, learning_rate=0.05, verbose=-1, n_jobs=-1,
                  random_state=42)
    model = lgb.LGBMClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)],
              callbacks=[lgb.early_stopping(20, verbose=False)])
    baseline = _logloss(model.predict_proba(X_vl), y_vl)
    imp = np.zeros(len(cols))
    rng = np.random.default_rng(42)
    for _ in range(n_repeats):
        for j in range(len(cols)):
            Xp = X_vl.copy(); rng.shuffle(Xp[:, j])
            imp[j] += _logloss(model.predict_proba(Xp), y_vl) - baseline
    imp /= n_repeats
    ranked = sorted(zip(cols, imp), key=lambda t: t[1], reverse=True)
    return [c for c, _ in ranked[:top_k]]


# ─────────────────────────────────────────────────────────────────────────────
if SELECTION_CACHE.exists():
    with open(SELECTION_CACHE) as f:
        _sel = json.load(f)
    SELECTED_FEATURES = _sel['features']
    print(f'Loaded {len(SELECTED_FEATURES)} selected features from cache.')
else:
    print('Running 4-stage feature selection on train split...')
    t0_sel = time.time()

    print(f'  Stage 1: variance + Spearman ({len(ALL_FEATURE_COLS)} → ?) ...')
    cols1, X1 = _stage1(X_sel, ALL_FEATURE_COLS)
    print(f'    → {len(cols1)} survivors')

    print(f'  Stage 2: MI ranking ({len(cols1)} → top {STAGE2_TOP_K}) ...')
    cols2, X2 = _stage2(X1, y_sel, cols1)
    print(f'    → {len(cols2)} survivors')

    print(f'  Stage 3: stability filter ({len(cols2)} → ?) ...')
    cols3, X3 = _stage3(X2, y_sel, cols2)
    print(f'    → {len(cols3)} survivors')

    print(f'  Stage 4: LGBM permutation pruning ({len(cols3)} → top {STAGE4_TOP_K}) ...')
    SELECTED_FEATURES = _stage4(X3, y_sel, cols3)
    print(f'    → {len(SELECTED_FEATURES)} final features')

    print(f'\nSelection done in {time.time()-t0_sel:.0f}s')
    with open(SELECTION_CACHE, 'w') as f:
        json.dump({'features': SELECTED_FEATURES,
                   'stage_counts': [len(ALL_FEATURE_COLS), len(cols1),
                                    len(cols2), len(cols3), len(SELECTED_FEATURES)]}, f, indent=2)

print(f'\nSelected features ({len(SELECTED_FEATURES)}):')
for i, f in enumerate(SELECTED_FEATURES, 1):
    is_v4 = ' [V4]' if f in V4_FEATURE_COLS else ''
    print(f'  {i:2d}. {f}{is_v4}')

Selection set: 53,581 bars × 220 features
Running 4-stage feature selection on train split...
  Stage 1: variance + Spearman (220 → ?) ...
    → 126 survivors
  Stage 2: MI ranking (126 → top 60) ...
    → 60 survivors
  Stage 3: stability filter (60 → ?) ...
    → 60 survivors
  Stage 4: LGBM permutation pruning (60 → top 20) ...
    → 20 final features

Selection done in 15s

Selected features (20):
   1. close_vs_true_vwap [V4]
   2. ret_2h
   3. close_vs_ema_7
   4. stoch_k_14
   5. rsi_vol_confirm
   6. hour_sin
   7. range_vs_atr
   8. candles_since_cross
   9. ret_3h
  10. close_vs_s1
  11. bull_streak
  12. vw_rsi_14
  13. candle_body
  14. macd_hist_5_13
  15. hl_position_24h
  16. ad_z_168h
  17. hurst_168h
  18. bb_squeeze_50
  19. ret_24h
  20. macd_divergence


In [5]:
# ── Feature subsets ───────────────────────────────────────────────────────────
# M4: full selected set (4-stage output)
# M1: V4 order-flow features that survived selection (or all V4 order-flow if none selected)
# M2: V1 features that survived selection
# M3: V4 regime/Hurst/ADF features that survived selection

_V4_ORDERFLOW = [
    'tfi_pct', 'tfi_z_24h', 'tfi_z_72h', 'tfi_z_168h', 'tfi_ema_12', 'tfi_ema_24',
    'avg_trade_size', 'avg_trade_size_z24',
    'true_vwap', 'close_vs_true_vwap',
    'taker_buy_price', 'taker_sell_price', 'taker_price_premium',
]
_V4_REGIME = [
    'hurst_24h', 'hurst_72h',
    'adf_tstat_168h', 'adf_pval_168h', 'adf_tstat_336h', 'adf_pval_336h',
    'adf_tstat_720h', 'adf_pval_720h',
    'bb_width_pct', 'vov_72h_pct', 'sideways_flag',
    'fracdiff_close_d0.2',
]

def _intersect_with_merged(cols):
    return [c for c in cols if c in merged.columns]

M1_FEATURES = _intersect_with_merged(
    [c for c in SELECTED_FEATURES if c in _V4_ORDERFLOW]
    or _intersect_with_merged(_V4_ORDERFLOW)  # fallback: all V4 order-flow
)
M2_FEATURES = _intersect_with_merged(
    [c for c in SELECTED_FEATURES if c in V1_FEATURE_COLS]
    or [c for c in SELECTED_FEATURES if c not in V4_FEATURE_COLS]
)
M3_FEATURES = _intersect_with_merged(
    [c for c in SELECTED_FEATURES if c in _V4_REGIME]
    or _intersect_with_merged(_V4_REGIME)  # fallback: all V4 regime
)
M4_FEATURES = _intersect_with_merged(SELECTED_FEATURES)

FEAT_SUBSETS = {
    'M1_order_flow':   M1_FEATURES,
    'M2_v1_selected':  M2_FEATURES,
    'M3_regime_dials': M3_FEATURES,
    'M4_omni':         M4_FEATURES,
}

for name, cols in FEAT_SUBSETS.items():
    v4_count = sum(c in V4_FEATURE_COLS for c in cols)
    print(f'{name}: {len(cols)} features  ({v4_count} V4)')
    if not cols:
        print(f'  ⚠ empty subset — check selection results')

M1_order_flow: 1 features  (1 V4)
M2_v1_selected: 19 features  (0 V4)
M3_regime_dials: 12 features  (12 V4)
M4_omni: 20 features  (1 V4)


---
## Phase C — Label Generation

- **FH:** 1-bar (1h) forward return > 0  
- **TBM:** Triple-Barrier — TP=1.5×ATR14, SL=1.5×ATR14, max_hold=24 bars (24h)

In [6]:
close_s = raw_df['close'].reindex(merged.index)
close_s.index = close_s.index.tz_localize(None) if close_s.index.tz else close_s.index

# ── FH label ──────────────────────────────────────────────────────────────────
y_fh = (close_s.pct_change(1).shift(-1) > 0).astype(int)
y_fh.name = 'label_fh'

# ── ATR for TBM ───────────────────────────────────────────────────────────────
hi_s = raw_df['high'].reindex(merged.index)
lo_s = raw_df['low'].reindex(merged.index)
hi_s.index = lo_s.index = close_s.index
tr     = pd.concat([
    hi_s - lo_s,
    (hi_s - close_s.shift(1)).abs(),
    (lo_s - close_s.shift(1)).abs(),
], axis=1).max(axis=1)
atr_1h = tr.ewm(span=14, adjust=False).mean()

# ── TBM label ─────────────────────────────────────────────────────────────────
SL_MULT  = 1.5
TP_MULT  = 1.5
MAX_HOLD = SEQ_LEN   # 24 bars = 24h

close_arr = close_s.values
atr_arr   = atr_1h.values
n         = len(close_arr)
y_tbm_arr = np.full(n, np.nan)

print(f'Computing TBM labels (n={n:,}, max_hold={MAX_HOLD}) ...')
t0 = time.time()
for i in range(n - MAX_HOLD):
    if np.isnan(atr_arr[i]) or atr_arr[i] == 0:
        continue
    entry = close_arr[i]
    tp    = entry + TP_MULT * atr_arr[i]
    sl    = entry - SL_MULT * atr_arr[i]
    for j in range(i + 1, min(i + MAX_HOLD + 1, n)):
        if close_arr[j] >= tp:   y_tbm_arr[i] = 1; break
        if close_arr[j] <= sl:   y_tbm_arr[i] = 0; break

y_tbm = pd.Series(y_tbm_arr, index=close_s.index, name='label_tbm')
print(f'TBM done in {time.time()-t0:.0f}s  |  '
      f'positive rate: {np.nanmean(y_tbm_arr):.3f}  |  '
      f'valid: {np.isfinite(y_tbm_arr).sum():,}')
print(f'FH  positive rate: {y_fh.mean():.3f}')

Computing TBM labels (n=74,366, max_hold=24) ...
TBM done in 0s  |  positive rate: 0.511  |  valid: 63,216
FH  positive rate: 0.509


In [7]:
df_base = merged[ALL_FEATURE_COLS].copy()
df_base['label_fh']  = y_fh.reindex(df_base.index)
df_base['label_tbm'] = y_tbm.reindex(df_base.index)
df_base['close']     = close_s.reindex(df_base.index)
df_base = df_base.dropna(subset=['close'])
df_base.index = df_base.index.tz_localize(None) if df_base.index.tz else df_base.index

train_mask = df_base.index <= TRAIN_END
val_mask   = (df_base.index > TRAIN_END) & (df_base.index <= VAL_END)
oos_mask   = df_base.index > VAL_END

df_tr  = df_base[train_mask].copy()
df_vl  = df_base[val_mask].copy()
df_oos = df_base[oos_mask].copy()

print(f'Train: {len(df_tr):,}  ({df_tr.index.min().date()} → {df_tr.index.max().date()})')
print(f'Val:   {len(df_vl):,}  ({df_vl.index.min().date()} → {df_vl.index.max().date()})')
print(f'OOS:   {len(df_oos):,}  ({df_oos.index.min().date()} → {df_oos.index.max().date()})')

Train: 44,799  (2017-11-15 → 2022-12-31)
Val:   8,759  (2022-12-31 → 2023-12-31)
OOS:   20,808  (2023-12-31 → 2026-05-16)


---
## Phase D — Normalization & Sequence Dataset

In [8]:
class SequenceDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray, seq_len: int, stride: int = 1):
        self.X       = X.astype(np.float32)
        self.y       = y
        self.seq_len = seq_len
        ends         = np.arange(seq_len - 1, len(X), stride)
        nan_frac     = np.array([np.isnan(X[e-seq_len+1:e+1]).mean() for e in ends])
        valid        = ~np.isnan(y[ends].astype(float)) & (nan_frac <= 0.10)
        self.ends    = ends[valid]

    def __len__(self): return len(self.ends)

    def __getitem__(self, idx):
        e = self.ends[idx]; s = e - self.seq_len + 1
        x = torch.from_numpy(np.nan_to_num(self.X[s:e+1], nan=0.0))
        return x, torch.tensor(int(self.y[e]), dtype=torch.long)


def make_loaders(feat_cols, target_col, batch=BATCH, seq_len=SEQ_LEN, stride=STRIDE):
    qt   = QuantileTransformer(n_quantiles=1000, output_distribution='normal', random_state=42)
    X_tr = qt.fit_transform(df_tr[feat_cols].fillna(0).values)
    X_vl = qt.transform(df_vl[feat_cols].fillna(0).values)
    y_tr = df_tr[target_col].values
    y_vl = df_vl[target_col].values
    ds_tr = SequenceDataset(X_tr, y_tr, seq_len, stride)
    ds_vl = SequenceDataset(X_vl, y_vl, seq_len, stride=1)
    return (
        DataLoader(ds_tr, batch_size=batch, shuffle=True,  drop_last=True),
        DataLoader(ds_vl, batch_size=batch, shuffle=False, drop_last=False),
        qt, len(feat_cols),
    )

print('SequenceDataset and make_loaders ready.')

SequenceDataset and make_loaders ready.


---
## Phase E — Mamba Architecture

In [9]:
class SelectiveSSM(nn.Module):
    """S6 selective scan: h_t = dA_t·h_{t-1} + dB_t·u_t,  y_t = C_t@h_t + D·u_t"""

    def __init__(self, d_inner: int, d_state: int, dt_rank: int):
        super().__init__()
        self.d_inner = d_inner
        self.d_state = d_state
        A_init       = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(d_inner, 1)
        self.A_log   = nn.Parameter(torch.log(A_init))           # (D, N)
        self.D       = nn.Parameter(torch.ones(d_inner))          # (D,)
        self.x_proj  = nn.Linear(d_inner, dt_rank + d_state * 2, bias=False)
        self.dt_proj = nn.Linear(dt_rank, d_inner, bias=True)
        nn.init.constant_(self.dt_proj.bias, math.log(math.expm1(1.0)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, L, D)  where D = d_inner
        B, L, D = x.shape
        A   = -torch.exp(self.A_log)                              # (D, N)
        dBC = self.x_proj(x)                                      # (B, L, dt_rank+2N)
        dt, B_ssm, C = dBC.split(
            [self.dt_proj.in_features, self.d_state, self.d_state], dim=-1
        )
        dt  = F.softplus(self.dt_proj(dt))                        # (B, L, D)
        dA  = torch.exp(dt.unsqueeze(-1) * A)                     # (B, L, D, N)
        dB  = dt.unsqueeze(-1) * B_ssm.unsqueeze(2)               # (B, L, D, N)

        h  = torch.zeros(B, D, self.d_state, device=x.device, dtype=x.dtype)  # (B, D, N)
        ys = []
        for t in range(L):
            # u_t: (B, D) → (B, D, 1)  for broadcasting with (B, D, N)
            h   = dA[:, t] * h + dB[:, t] * x[:, t, :].unsqueeze(-1)  # (B, D, N)
            # C_t: (B, N) → (B, 1, N)  for broadcasting with (B, D, N)
            y_t = (h * C[:, t, :].unsqueeze(1)).sum(-1)                 # (B, D)
            ys.append(y_t)

        y = torch.stack(ys, dim=1)   # (B, L, D)
        return y + x * self.D        # residual skip, self.D broadcasts over (B, L, D)


class MambaBlock(nn.Module):
    def __init__(self, d_model: int, d_state: int = 16, d_conv: int = 4, expand: int = 2):
        super().__init__()
        d_inner        = d_model * expand
        dt_rank        = max(1, d_model // 16)
        self.in_proj   = nn.Linear(d_model, d_inner * 2, bias=False)
        self.conv1d    = nn.Conv1d(d_inner, d_inner, d_conv, padding=d_conv - 1,
                                   groups=d_inner, bias=True)
        self.ssm       = SelectiveSSM(d_inner, d_state, dt_rank)
        self.out_proj  = nn.Linear(d_inner, d_model, bias=False)
        self.norm      = nn.LayerNorm(d_model)

    def forward(self, x):
        residual = x
        x        = self.norm(x)
        xz       = self.in_proj(x)
        x_in, z  = xz.chunk(2, dim=-1)
        # causal conv: (B, d_inner, L) → trim to original L → (B, L, d_inner)
        x_in = self.conv1d(x_in.transpose(1, 2))[..., :x.shape[1]].transpose(1, 2)
        x_in = F.silu(x_in)
        return self.out_proj(self.ssm(x_in) * F.silu(z)) + residual


class MambaClassifier(nn.Module):
    def __init__(self, n_features: int, d_model: int = 64, n_layers: int = 2,
                 d_state: int = 16, d_conv: int = 4, expand: int = 2,
                 n_classes: int = 2, dropout: float = 0.1):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.blocks     = nn.ModuleList([
            MambaBlock(d_model, d_state, d_conv, expand) for _ in range(n_layers)
        ])
        self.dropout    = nn.Dropout(dropout)
        self.head       = nn.Linear(d_model, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input_proj(x)
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.dropout(x[:, -1, :]))


print('MambaClassifier ready — d_model=64  n_layers=2  d_state=16  d_conv=4  expand=2')

# Quick shape smoke-test
_m = MambaClassifier(n_features=5).to(DEVICE)
_x = torch.randn(4, SEQ_LEN, 5).to(DEVICE)
assert _m(_x).shape == (4, 2), f'Unexpected output shape: {_m(_x).shape}'
del _m, _x
print('Shape smoke-test passed.')

MambaClassifier ready — d_model=64  n_layers=2  d_state=16  d_conv=4  expand=2
Shape smoke-test passed.


In [10]:
class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0, alpha: torch.Tensor = None):
        super().__init__()
        self.gamma = gamma
        self.register_buffer('alpha', alpha)

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none', weight=self.alpha)
        return ((1 - torch.exp(-ce)) ** self.gamma * ce).mean()


def make_focal_loss(y_train, gamma=FOCAL_GAMMA, device=DEVICE):
    counts = np.bincount(y_train[~np.isnan(y_train)].astype(int), minlength=2)
    alpha  = torch.tensor(1.0 / (counts + 1), dtype=torch.float32).to(device)
    return FocalLoss(gamma=gamma, alpha=alpha / alpha.sum())


def train_one_epoch(model, loader, criterion, opt, device):
    model.train()
    total = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, device):
    model.eval()
    all_p, all_y = [], []
    for xb, yb in loader:
        p = torch.softmax(model(xb.to(device)), dim=-1)[:, 1].cpu().numpy()
        all_p.append(p); all_y.append(yb.numpy())
    p = np.concatenate(all_p); y = np.concatenate(all_y)
    return p, y, (roc_auc_score(y, p) if len(np.unique(y)) >= 2 else 0.5)

print('FocalLoss + training utilities ready.')

FocalLoss + training utilities ready.


---
## Phase F — Feature Attribution

In [11]:
def compute_sequence_permutation_importance(
    model, dataloader, base_auc, device, feature_names, n_repeats=3
):
    model.eval()
    all_x, all_y = [], []
    with torch.no_grad():
        for xb, yb in dataloader:
            all_x.append(xb); all_y.append(yb)
    X_clean = torch.cat(all_x, dim=0).to(device)
    Y_true  = torch.cat(all_y, dim=0).cpu().numpy()
    importances = {}
    for f_idx, fname in enumerate(feature_names):
        drops = []
        for _ in range(n_repeats):
            Xp = X_clean.clone()
            Xp[:, :, f_idx] = Xp[torch.randperm(Xp.shape[0]), :, f_idx]
            preds = []
            with torch.no_grad():
                for chunk in torch.split(Xp, 256, dim=0):
                    preds.append(torch.softmax(model(chunk), dim=-1)[:, 1].cpu())
            Yp = torch.cat(preds).numpy()
            if len(np.unique(Y_true)) >= 2:
                drops.append(base_auc - roc_auc_score(Y_true, Yp))
        importances[fname] = float(np.mean(drops)) if drops else 0.0
    return importances


def compute_integrated_gradients(model, x_input, device, baseline=None, n_steps=50, last_n_bars=4):
    model.eval()
    x_input = x_input.to(device)
    if baseline is None:
        baseline = torch.zeros_like(x_input).to(device)
    grads = torch.zeros_like(x_input[0])
    for alpha in torch.linspace(0, 1, n_steps, device=device):
        interp = (baseline + alpha * (x_input - baseline)).requires_grad_(True)
        torch.softmax(model(interp), dim=-1)[0, 1].backward()
        grads += interp.grad[0].detach()
    ig = ((x_input[0] - baseline[0]).detach() * grads / n_steps).abs()
    return ig[-last_n_bars:].mean(dim=0).cpu().numpy()

print('Attribution functions ready.')

Attribution functions ready.


---
## Phase G — Training (4 Subsets × 2 Targets = 8 Runs)

In [ ]:
RESULTS = []

for subset_name, feat_cols in FEAT_SUBSETS.items():
    if not feat_cols:
        print(f'SKIP {subset_name}: empty feature list')
        continue
    for target_col in ['label_fh', 'label_tbm']:
        run_id      = f'{subset_name}__{target_col}'
        cache_probs = ARTS_DIR / f'probs_{run_id}.npy'
        cache_meta  = ARTS_DIR / f'meta_{run_id}.json'

        if cache_probs.exists() and cache_meta.exists():
            print(f'[{run_id}] Loading from cache.')
            val_probs = np.load(cache_probs)
            with open(cache_meta) as f:
                meta = json.load(f)
            RESULTS.append({'run_id': run_id, 'feat_key': subset_name,
                            'target': target_col, 'val_probs': val_probs, **meta})
            continue

        print(f'\n{"="*65}')
        print(f'RUN: {run_id}  ({len(feat_cols)} features)')
        print(f'{"="*65}')

        tr_loader, vl_loader, qt, n_feat = make_loaders(feat_cols, target_col)
        if len(tr_loader) == 0:
            print('  SKIP: empty training loader'); continue

        model     = MambaClassifier(n_features=n_feat).to(DEVICE)
        print(f'  Parameters: {sum(p.numel() for p in model.parameters()):,}')

        y_tr_vals = df_tr[target_col].dropna().values.astype(int)
        criterion = make_focal_loss(y_tr_vals)
        opt       = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
        sched     = SequentialLR(opt,
                        [LinearLR(opt, start_factor=0.1, end_factor=1.0, total_iters=5),
                         CosineAnnealingLR(opt, T_max=EPOCHS - 5, eta_min=1e-6)],
                        milestones=[5])

        best_auc = best_epoch = 0
        patience = 10
        train_aucs, val_aucs = [], []
        t_run = time.time()

        for epoch in range(1, EPOCHS + 1):
            tr_loss             = train_one_epoch(model, tr_loader, criterion, opt, DEVICE)
            _, _, tr_auc        = eval_epoch(model, tr_loader, DEVICE)
            vl_probs, vl_y, vl_auc = eval_epoch(model, vl_loader, DEVICE)
            sched.step()
            train_aucs.append(tr_auc); val_aucs.append(vl_auc)

            if vl_auc > best_auc:
                best_auc = vl_auc; best_epoch = epoch
                best_probs = vl_probs.copy()
                torch.save(model.state_dict(), ARTS_DIR / f'model_{run_id}.pt')

            if epoch % 5 == 0 or epoch == 1:
                print(f'  Ep {epoch:3d}: loss={tr_loss:.4f}  '
                      f'tr={tr_auc:.4f}  val={vl_auc:.4f}  best={best_auc:.4f}')

            if epoch - best_epoch >= patience:
                print(f'  Early stop ep {epoch}  (best={best_auc:.4f} @ {best_epoch})')
                break

        elapsed = time.time() - t_run
        print(f'  Best val AUC: {best_auc:.4f} @ ep {best_epoch}  ({elapsed:.0f}s)')

        np.save(cache_probs, best_probs)
        meta = {'best_auc': float(best_auc), 'best_epoch': int(best_epoch),
                'train_aucs': train_aucs, 'val_aucs': val_aucs,
                'n_features': int(n_feat), 'n_train_seqs': len(tr_loader.dataset),
                'n_val_seqs': len(vl_loader.dataset), 'elapsed_s': float(elapsed),
                'feat_cols': feat_cols}
        with open(cache_meta, 'w') as f:
            json.dump(meta, f, indent=2)
        RESULTS.append({'run_id': run_id, 'feat_key': subset_name,
                        'target': target_col, 'val_probs': best_probs, **meta})

print(f'\nTraining complete: {len(RESULTS)} runs finished')


RUN: M1_order_flow__label_fh  (1 features)
  Parameters: 65,794


In [ ]:
rows = [{'Run': r['run_id'].replace('__', ' | '), 'Features': r['n_features'],
         'Val AUC': round(r['best_auc'], 4), 'Best Ep': r['best_epoch'],
         'Train Seqs': r['n_train_seqs'], 'Val Seqs': r['n_val_seqs']}
        for r in RESULTS]
print(pd.DataFrame(rows).sort_values('Val AUC', ascending=False).to_string(index=False))
best_run = max(RESULTS, key=lambda r: r['best_auc']) if RESULTS else None
if best_run:
    print(f'\nBest: {best_run["run_id"]}  AUC={best_run["best_auc"]:.4f}')

---
## Phase H — Feature Attribution

In [ ]:
PERM_IMP_RESULTS = {}

for r in RESULTS:
    run_id     = r['run_id']
    feat_cols  = r['feat_cols']
    target_col = r['target']
    cache_perm = ARTS_DIR / f'perm_imp_{run_id}.json'

    if cache_perm.exists():
        with open(cache_perm) as f:
            PERM_IMP_RESULTS[run_id] = json.load(f)
        print(f'[{run_id}] loaded from cache.')
        continue

    model_path = ARTS_DIR / f'model_{run_id}.pt'
    if not model_path.exists():
        print(f'[{run_id}] model not found — skipping.'); continue

    _, vl_loader, _, n_feat = make_loaders(feat_cols, target_col, stride=1)
    model = MambaClassifier(n_features=n_feat).to(DEVICE)
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))

    print(f'[{run_id}] Computing permutation importance...')
    imp = compute_sequence_permutation_importance(
        model, vl_loader, base_auc=r['best_auc'],
        device=DEVICE, feature_names=feat_cols, n_repeats=3,
    )
    PERM_IMP_RESULTS[run_id] = imp
    with open(cache_perm, 'w') as f:
        json.dump(imp, f, indent=2)
    print(f'  Top 5: {sorted(imp, key=imp.get, reverse=True)[:5]}')

In [ ]:
IG_RESULTS = {}

for r in RESULTS[:2]:
    run_id     = r['run_id']
    feat_cols  = r['feat_cols']
    target_col = r['target']
    cache_ig   = ARTS_DIR / f'ig_{run_id}.npy'

    if cache_ig.exists():
        IG_RESULTS[run_id] = (np.load(cache_ig), feat_cols)
        print(f'[{run_id}] IG loaded from cache.'); continue

    model_path = ARTS_DIR / f'model_{run_id}.pt'
    if not model_path.exists(): continue

    _, vl_loader, _, n_feat = make_loaders(feat_cols, target_col, stride=1)
    model = MambaClassifier(n_features=n_feat).to(DEVICE)
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.eval()

    for xb, _ in vl_loader:
        x_example = xb[0:1].to(DEVICE); break

    print(f'[{run_id}] Computing IG (n_steps=50, last_4_bars)...')
    ig_scores = compute_integrated_gradients(model, x_example, DEVICE,
                                             n_steps=50, last_n_bars=4)
    IG_RESULTS[run_id] = (ig_scores, feat_cols)
    np.save(cache_ig, ig_scores)
    print(f'  Done.')

---
## Phase I — Visualizations

In [ ]:
n_runs = len(RESULTS)
if n_runs > 0:
    cols_ = min(4, n_runs); rows_ = math.ceil(n_runs / cols_)
    fig, axes = plt.subplots(rows_, cols_, figsize=(cols_ * 4, rows_ * 3.5), squeeze=False)
    for ax, r in zip(axes.flatten(), RESULTS):
        ep = range(1, len(r['train_aucs']) + 1)
        ax.plot(ep, r['train_aucs'], color='#2196F3', lw=1.2, label='Train')
        ax.plot(ep, r['val_aucs'],   color='#FF6F00', lw=1.2, label='Val')
        ax.axhline(r['best_auc'], color='green', lw=0.8, ls='--',
                   label=f'Best={r["best_auc"]:.4f}@{r["best_epoch"]}')
        ax.set_title(r['run_id'].replace('__', '\n').replace('label_', ''), fontsize=8)
        ax.legend(fontsize=7); ax.set_ylim(0.45, 0.75); ax.grid(alpha=0.3)
    for ax in axes.flatten()[n_runs:]: ax.set_visible(False)
    fig.suptitle('Mamba v2 — Training Curves', fontsize=12, fontweight='bold')
    fig.tight_layout()
    fig.savefig(ARTS_DIR / 'training_curves.png', dpi=130, bbox_inches='tight')
    plt.show()

In [ ]:
if PERM_IMP_RESULTS and best_run:
    all_feats  = sorted(set(f for imp in PERM_IMP_RESULTS.values() for f in imp))
    imp_matrix = pd.DataFrame(
        {rid: [imp.get(f, 0) for f in all_feats] for rid, imp in PERM_IMP_RESULTS.items()},
        index=all_feats,
    )
    fig, axes = plt.subplots(1, 2, figsize=(14, max(6, len(all_feats) * 0.3)))
    sns.heatmap(imp_matrix, ax=axes[0], cmap='RdYlGn', center=0, annot=True, fmt='.4f',
                linewidths=0.5, cbar_kws={'label': 'AUC Drop'})
    axes[0].set_title('Permutation Importance — AUC Drop per Feature')
    axes[0].tick_params(labelsize=7)
    best_imp  = PERM_IMP_RESULTS.get(best_run['run_id'], {})
    sorted_f  = sorted(best_imp, key=best_imp.get, reverse=True)
    axes[1].barh(range(len(sorted_f)), [best_imp[f] for f in sorted_f],
                 color=['#2E7D32' if best_imp[f] > 0 else '#C62828' for f in sorted_f])
    axes[1].set_yticks(range(len(sorted_f)))
    axes[1].set_yticklabels(sorted_f, fontsize=8)
    axes[1].invert_yaxis(); axes[1].set_xlabel('AUC Drop')
    axes[1].set_title(f'Best: {best_run["run_id"]}')
    fig.tight_layout()
    fig.savefig(ARTS_DIR / 'perm_importance.png', dpi=130, bbox_inches='tight')
    plt.show()

In [ ]:
for run_id, (ig_scores, feat_cols) in IG_RESULTS.items():
    sorted_idx = np.argsort(ig_scores)[::-1]
    fig, ax = plt.subplots(figsize=(10, max(5, len(feat_cols) * 0.35)))
    ax.barh(range(len(feat_cols)), ig_scores[sorted_idx],
            color=['#2196F3' if ig_scores[i] > np.median(ig_scores) else '#9E9E9E'
                   for i in sorted_idx])
    ax.set_yticks(range(len(feat_cols)))
    ax.set_yticklabels([feat_cols[i] for i in sorted_idx], fontsize=8)
    ax.invert_yaxis(); ax.set_xlabel('Mean |IG| over last 4 bars')
    ax.set_title(f'Integrated Gradients — {run_id}', fontweight='bold')
    fig.tight_layout()
    fig.savefig(ARTS_DIR / f'ig_{run_id}.png', dpi=130, bbox_inches='tight')
    plt.show()

---
## Phase J — Save Results

In [ ]:
out = {
    'notebook': '03_mamba_omni_0fee_v2', 'version': 'v2',
    'interval': INTERVAL, 'created': pd.Timestamp.now().isoformat(),
    'device': str(DEVICE), 'seq_len': SEQ_LEN, 'stride': STRIDE, 'epochs': EPOCHS,
    'feature_pool': {'v1': len(V1_FEATURE_COLS), 'v4': len(V4_FEATURE_COLS),
                     'total': len(ALL_FEATURE_COLS)},
    'selected_features': SELECTED_FEATURES,
    'results': [{k: v for k, v in r.items() if k not in ('val_probs','train_aucs','val_aucs')}
                for r in RESULTS],
    'permutation_importance': PERM_IMP_RESULTS,
    'integrated_gradients': {
        rid: {'scores': ig.tolist(), 'features': feats}
        for rid, (ig, feats) in IG_RESULTS.items()
    },
}
out_path = ARTS_DIR / 'feature_importance.json'
with open(out_path, 'w') as f:
    json.dump(out, f, indent=2, default=str)
print(f'Saved: {out_path}')
print('\n── Final Summary ─────────────────────────────────────────')
for r in sorted(RESULTS, key=lambda x: x['best_auc'], reverse=True):
    print(f'  {r["run_id"]:<55s}  AUC={r["best_auc"]:.4f}  ep={r["best_epoch"]}')